# ProcureCompare AI

**Intelligent Product Sourcing and Price Comparison Agent**

In [ ]:
# Run this cell once after selecting the Python 3.12 (ai-agent) kernel.
%pip install -q -r requirements.txt

In [ ]:
import sys

# Fail early when the selected notebook kernel uses an unsupported Python version.
print(f"Python: {sys.version.split()[0]}")
print(f"Interpreter: {sys.executable}")
assert sys.version_info[:2] in {(3, 10), (3, 11), (3, 12)}, (
    "This project requires Python 3.10-3.12. Select the Python 3.12 (ai-agent) kernel."
)

In [ ]:
# CrewAI handles reasoning and report writing; Python controls external API calls.
from crewai import Agent, Task, Crew, Process, LLM
import agentops
from agentops.sdk.exporters import AuthenticatedOTLPExporter
from agentops.sdk import processors as agentops_processors
from pydantic import BaseModel, Field
from typing import List, Optional
from tavily import TavilyClient
from scrapegraph_py import ScrapeGraphAI, FetchConfig
from dotenv import load_dotenv
from pathlib import Path
from types import SimpleNamespace
from urllib.parse import urlparse, urlunparse

import os
import json

# Support kernels started either from this project folder or from the repository root.
env_candidates = [Path.cwd() / ".env", Path.cwd() / "agents/procurecompare-ai/.env"]
env_path = next((path for path in env_candidates if path.is_file()), None)
if env_path is None:
    raise FileNotFoundError("Local .env file not found. Copy .env.example to .env first.")

# Resolve stable project/repository paths instead of depending on the kernel working directory.
project_dir = env_path.parent
repository_root = project_dir.parents[1] if project_dir.parent.name == "agents" else project_dir

# Replace stale values inherited by the VS Code/Jupyter server.
load_dotenv(dotenv_path=env_path, override=True)

def get_secret(
    env_name: str,
    *,
    required: bool = True,
) -> Optional[str]:
    """Read a secret from the macOS process environment or local .env file."""
    value = os.getenv(env_name)
    if value and not value.startswith("your_"):
        return value
    if not required:
        return None

    raise RuntimeError(
        f"Missing {env_name}. Add it to the local .env file, then rerun this cell."
    )

In [ ]:
# OpenAI is required by every CrewAI agent; AgentOps tracing is optional.
openai_api_key = get_secret("OPENAI_API_KEY")
if not openai_api_key.startswith("sk-"):
    raise RuntimeError("OPENAI_API_KEY has an invalid format. Use an OpenAI secret key beginning with 'sk-'.")
agentops_api_key = get_secret("AGENTOPS_API_KEY", required=False)
os.environ["OPENAI_API_KEY"] = openai_api_key

# AgentOps 0.4.x expects private OpenTelemetry attributes removed in OTEL 1.45.
# Restore small adapters for the new internal HTTP client without downgrading OTEL.
if not hasattr(AuthenticatedOTLPExporter, "_procurecompare_original_init"):
    AuthenticatedOTLPExporter._procurecompare_original_init = AuthenticatedOTLPExporter.__init__

def _initialize_compatible_agentops_exporter(self, *args, **kwargs):
    AuthenticatedOTLPExporter._procurecompare_original_init(self, *args, **kwargs)
    if not hasattr(self, "_headers"):
        self._headers = {}
    if not hasattr(self, "_session") and hasattr(self, "_client"):
        self._session = SimpleNamespace(headers=self._client._headers)

def _prepare_agentops_headers(self, headers=None):
    prepared_headers = dict(getattr(self, "_headers", {}))
    filtered_headers = self._filter_user_headers(headers)
    if filtered_headers:
        prepared_headers.update(filtered_headers)
    jwt_token = self._get_current_jwt()
    if jwt_token:
        prepared_headers["Authorization"] = f"Bearer {jwt_token}"
    return prepared_headers

AuthenticatedOTLPExporter.__init__ = _initialize_compatible_agentops_exporter
AuthenticatedOTLPExporter._prepare_headers = _prepare_agentops_headers

# AgentOps separately uploads captured console logs through an endpoint that can return
# 401 on the free plan. Traces and LLM spans still use the working OTLP exporter.
def _skip_optional_agentops_logfile_upload(_trace_id):
    return None

agentops_processors.upload_logfile = _skip_optional_agentops_logfile_upload

if agentops_api_key:
    os.environ["AGENTOPS_API_KEY"] = agentops_api_key
    try:
        agentops.init(
            api_key=agentops_api_key,
            auto_start_session=False,
            skip_auto_end_session=True,
            default_tags=["crewai"],
        )
        print("AgentOps enabled.")
    except Exception as error:
        print(f"AgentOps disabled after initialization error: {error}")
        agentops_api_key = None
else:
    print("AgentOps disabled (optional key not configured).")

print("API configuration loaded successfully.")

In [ ]:
from IPython.display import Markdown, display
from agentops.helpers.dashboard import get_trace_url

def start_agentops_trace():
    """Start a fresh trace for each workflow run and display its dashboard link."""
    if not agentops_api_key:
        print("AgentOps is disabled.")
        return None

    try:
        trace = agentops.start_trace(
            trace_name="crewai-procurement",
            tags=["crewai"],
        )
    except Exception as error:
        print(f"AgentOps warning: unable to start trace: {error}")
        return None
    if trace:
        replay_url = get_trace_url(trace.span)
        display(Markdown(f"🖇 **AgentOps — [Session Replay]({replay_url})**"))
    return trace

In [ ]:
# Store every artifact in the repository-level output directory requested by this project.
output_dir = repository_root / "ai-agent-output"
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Workflow outputs: {output_dir}")

tavily_api_key = get_secret("TAVILY_API_KEY")
scrapegraph_api_key = get_secret("SCRAPEGRAPH_API_KEY")
os.environ.setdefault("SGAI_TIMEOUT", "60")

# Use deterministic generation and initialize the two external data providers.
basic_llm = LLM(model="gpt-4o", temperature=0)
search_client = TavilyClient(api_key=tavily_api_key)
scrape_client = ScrapeGraphAI(api_key=scrapegraph_api_key)

In [ ]:
# Limit query generation and provide business context to the report writer.
no_keywords = 10

about_company = "ProcureCompare AI helps organizations source products, compare prices, and make informed purchasing decisions."

## Setup Agents

### Agent: A

In [ ]:
# Enforce a predictable JSON structure for the query-planning agent output.
class SuggestedSearchQueries(BaseModel):
    queries: List[str] = Field(..., title="Suggested search queries to be passed to the search engine",
                               min_length=1, max_length=no_keywords)

# Agent A converts the purchasing request into targeted e-commerce searches.
search_queries_recommendation_agent = Agent(
    role="Search Queries Recommendation Agent",
    goal="\n".join([
                "To provide a list of suggested search queries to be passed to the search engine.",
                "The queries must be varied and looking for specific items."
            ]),
    backstory="The agent is designed to help in looking for products by providing a list of suggested search queries to be passed to the search engine based on the context provided.",
    llm=basic_llm,
    verbose=True,
)

# The task writes its validated result to disk for inspection and reuse.
search_queries_recommendation_task = Task(
    description="\n".join([
        "The buyer is looking to purchase {product_name} using a value-for-money strategy.",
        "The campany target any of these websites to buy from: {websites_list}",
        "The company wants to reach all available proucts on the internet to be compared later in another stage.",
        "The stores must sell the product in {country_name}",
        "Generate at maximum {no_keywords} queries.",
        "The search keywords must be in {language} language.",
        "Search keywords must contains specific brands, types or technologies. Avoid general keywords.",
        "The search query must reach an ecommerce webpage for product, and not a blog or listing page."
    ]),
    expected_output="A JSON object containing a list of suggested search queries.",
    output_json=SuggestedSearchQueries,
    output_file=os.path.join(output_dir, "step_1_suggested_search_queries.json"),
    agent=search_queries_recommendation_agent
)

### Deterministic Tavily Search Pipeline

In [ ]:
# Normalize and validate every result returned by Tavily.
class SingleSearchResult(BaseModel):
    title: str
    url: str = Field(..., title="the page url")
    content: str
    score: float
    search_query: str

class AllSearchResults(BaseModel):
    results: List[SingleSearchResult]

ALLOWED_STORE_DOMAINS = ("amazon.eg", "jumia.com.eg", "noon.com")

def canonical_product_url(url: str) -> str:
    """Remove tracking parameters so the same product is not selected twice."""
    parsed = urlparse(url.strip())
    return urlunparse((parsed.scheme.lower(), parsed.netloc.lower(), parsed.path.rstrip("/"), "", "", ""))

def is_product_page(url: str) -> bool:
    """Accept only recognizable single-product pages from the configured stores."""
    parsed = urlparse(url)
    host = parsed.netloc.lower().removeprefix("www.")
    path = parsed.path.lower()
    if not any(host == domain or host.endswith(f".{domain}") for domain in ALLOWED_STORE_DOMAINS):
        return False
    if host.endswith("amazon.eg"):
        return any(pattern in path for pattern in ("/dp/", "/gp/product/", "/gp/aw/d/"))
    if host.endswith("jumia.com.eg"):
        return path.endswith(".html")
    if host.endswith("noon.com"):
        is_egypt_store = path.startswith("/egypt-en/") or path.startswith("/egypt-ar/")
        return is_egypt_store and ("/p/" in path or path.rstrip("/").endswith("/p"))
    return False

def run_product_searches(
    queries: List[str],
    score_threshold: float = 0.10,
    max_results_per_query: int = 5,
    max_total_results: int = 10,
) -> AllSearchResults:
    """Run every query once, validate results, deduplicate URLs, and keep query diversity."""
    unique_results = {}
    first_result_per_query = []
    search_errors = []

    for query_index, query in enumerate(queries, start=1):
        print(f"[Tavily {query_index}/{len(queries)}] {query}")
        try:
            response = search_client.search(
                query=query,
                search_depth="advanced",
                max_results=max_results_per_query,
                include_domains=list(ALLOWED_STORE_DOMAINS),
                include_raw_content=False,
                timeout=30,
            )
        except Exception as error:
            search_errors.append({"query": query, "error": str(error)})
            continue
        accepted_for_query = []
        for item in response.get("results", []):
            score = float(item.get("score") or 0)
            url = item.get("url") or ""
            if score < score_threshold or not is_product_page(url):
                continue
            canonical_url = canonical_product_url(url)
            if canonical_url in unique_results:
                continue
            result = SingleSearchResult(
                title=item.get("title") or "Untitled product",
                url=canonical_url,
                content=item.get("content") or "",
                score=score,
                search_query=query,
            )
            unique_results[canonical_url] = result
            accepted_for_query.append(result)
        if accepted_for_query:
            first_result_per_query.append(max(accepted_for_query, key=lambda result: result.score))

    Path(output_dir, "step_2_search_errors.json").write_text(
        json.dumps(search_errors, indent=2), encoding="utf-8"
    )
    if not unique_results:
        raise RuntimeError("Tavily returned no valid product-detail pages above the score threshold.")

    selected = []
    selected_urls = set()
    for result in first_result_per_query:
        if result.url not in selected_urls and len(selected) < max_total_results:
            selected.append(result)
            selected_urls.add(result.url)
    for result in sorted(unique_results.values(), key=lambda result: result.score, reverse=True):
        if result.url not in selected_urls and len(selected) < max_total_results:
            selected.append(result)
            selected_urls.add(result.url)

    payload = AllSearchResults(results=selected)
    Path(output_dir, "step_2_search_results.json").write_text(
        payload.model_dump_json(indent=2), encoding="utf-8"
    )
    return payload

### Deterministic ScrapeGraph Extraction Pipeline

In [ ]:
# Define only factual fields that can be extracted from one product page.
class ProductSpec(BaseModel):
    specification_name: str
    specification_value: str

class SingleExtractedProduct(BaseModel):
    page_url: str = Field(..., title="The original url of the product page")
    product_title: str = Field(..., title="The title of the product")
    product_image_url: Optional[str] = Field(default=None, title="The url of the product image")
    product_url: str = Field(..., title="The url of the product")
    product_current_price: float = Field(..., gt=0, title="The current price of the product")
    product_original_price: Optional[float] = Field(title="The original price of the product before discount. Set to None if no discount", default=None)
    product_discount_percentage: Optional[float] = Field(title="The discount percentage of the product. Set to None if no discount", default=None)
    product_specs: List[ProductSpec] = Field(default_factory=list, max_length=5)
    source_search_query: str = ""
    source_search_score: float = 0.0

class AllExtractedProducts(BaseModel):
    products: List[SingleExtractedProduct]

def run_product_scraping(
    search_results: AllSearchResults,
    max_products: int = 5,
) -> AllExtractedProducts:
    """Extract products one by one, reject invalid data, and record failures separately."""
    products = []
    errors = []
    extraction_schema = SingleExtractedProduct.model_json_schema()
    products_path = Path(output_dir, "step_3_extracted_products.json")
    errors_path = Path(output_dir, "step_3_scrape_errors.json")

    def save_extraction_checkpoint():
        products_path.write_text(
            AllExtractedProducts(products=products).model_dump_json(indent=2), encoding="utf-8"
        )
        errors_path.write_text(json.dumps(errors, indent=2), encoding="utf-8")

    save_extraction_checkpoint()

    for result_index, search_result in enumerate(search_results.results, start=1):
        if len(products) >= max_products:
            break
        print(f"[ScrapeGraph {result_index}/{len(search_results.results)}] {search_result.url}")
        try:
            result = scrape_client.extract(
                url=search_result.url,
                prompt=(
                    "Extract only factual product details visible on this single-product page. "
                    "Use the listed current price as a number without currency symbols. "
                    "Use null for missing optional values and never invent data."
                ),
                schema=extraction_schema,
                fetch_config=FetchConfig(mode="auto", timeout=30000),
            )
            if result.status == "error" or not result.data:
                raise RuntimeError(result.error or "ScrapeGraph returned no data")
            details = dict(result.data.json_data or {})
            details["page_url"] = search_result.url
            details["product_url"] = search_result.url
            details["source_search_query"] = search_result.search_query
            details["source_search_score"] = search_result.score
            products.append(SingleExtractedProduct.model_validate(details))
            print(f"  Extracted: {products[-1].product_title}")
        except Exception as error:
            errors.append({"url": search_result.url, "error": str(error)})
            print(f"  Skipped: {error}")
        finally:
            save_extraction_checkpoint()

    if not products:
        raise RuntimeError("No product page could be extracted. See step_3_scrape_errors.json.")

    payload = AllExtractedProducts(products=products)
    save_extraction_checkpoint()
    return payload

### Agent: D

In [ ]:
# Agent D turns the collected product data into the final human-readable report.
procurement_report_author_agent = Agent(
    role="Procurement Report Author Agent",
    goal="To generate a professional HTML page for the procurement report",
    backstory="The agent is designed to assist in generating a professional HTML page for the procurement report after looking into a list of products.",
    llm=basic_llm,
    verbose=True,
)

procurement_report_author_task = Task(
    description="\n".join([
        "Generate a professional procurement report using ONLY the validated product JSON below.",
        "Never add a product, price, specification, store, claim, or URL absent from this JSON.",
        "If the data is incomplete, explicitly disclose that limitation instead of guessing.",
        "Every product table row must map to exactly one supplied product.",
        "Validated products JSON:",
        "{products_json}",
        "Company context: {company_context}",
        "You have to use Bootstrap CSS framework for a better UI.",
        "The report should be structured with the following sections:",
        "1. Executive Summary: A brief overview of the procurement process and key findings.",
        "2. Introduction: An introduction to the purpose and scope of the report.",
        "3. Methodology: A description of the methods used to gather and compare prices.",
        "4. Findings: Detailed comparison of prices from different websites, including tables and charts.",
        "5. Analysis: An analysis of the findings, highlighting any significant trends or observations.",
        "6. Recommendations: Suggestions for procurement based on the analysis.",
        "7. Conclusion: A summary of the report and final thoughts.",
        "8. Appendices: Include the supplied source URLs and search provenance.",
        "Return raw HTML beginning with <!DOCTYPE html>; do not use Markdown code fences.",
    ]),

    expected_output="A complete raw HTML document grounded only in the supplied JSON.",
    output_file=os.path.join(output_dir, "step_4_procurement_report.html"),
    agent=procurement_report_author_agent,
)

## Run the AI Crew

In [ ]:
# Agents reason and write; deterministic Python functions own all external data calls.
query_crew = Crew(
    agents=[search_queries_recommendation_agent],
    tasks=[search_queries_recommendation_task],
    process=Process.sequential,
)

report_crew = Crew(
    agents=[procurement_report_author_agent],
    tasks=[procurement_report_author_task],
    process=Process.sequential,
)

def parse_suggested_queries(crew_output) -> SuggestedSearchQueries:
    """Read CrewAI structured output without trusting free-form text."""
    if crew_output.pydantic:
        return SuggestedSearchQueries.model_validate(crew_output.pydantic)
    if crew_output.json_dict:
        return SuggestedSearchQueries.model_validate(crew_output.json_dict)
    output_path = Path(output_dir, "step_1_suggested_search_queries.json")
    return SuggestedSearchQueries.model_validate_json(output_path.read_text(encoding="utf-8"))

def remove_stale_artifacts() -> None:
    """Prevent files from a previous run from being mistaken for current results."""
    for filename in (
        "step_1_suggested_search_queries.json",
        "step_2_search_results.json",
        "step_2_search_errors.json",
        "step_3_extracted_products.json",
        "step_3_scrape_errors.json",
        "step_4_procurement_report.html",
    ):
        Path(output_dir, filename).unlink(missing_ok=True)

def save_clean_html(raw_html: str) -> Path:
    """Reject prose-only output and remove accidental Markdown fences."""
    text = raw_html.strip()
    if text.startswith("```"):
        text = text.split("\n", 1)[1] if "\n" in text else text
        if text.rstrip().endswith("```"):
            text = text.rstrip()[:-3].rstrip()
    report_path = Path(output_dir, "step_4_procurement_report.html")
    lowered = text.lower()
    start = lowered.find("<!doctype html")
    if start < 0:
        start = lowered.find("<html")
    end = lowered.rfind("</html>")
    if start < 0 or end < start:
        report_path.unlink(missing_ok=True)
        raise RuntimeError("The report agent did not return a valid HTML document.")
    report_path.write_text(text[start:end + len("</html>")], encoding="utf-8")
    return report_path

def run_procurement_workflow(inputs: dict) -> dict:
    """Execute the validated four-stage procurement workflow."""
    remove_stale_artifacts()
    trace = start_agentops_trace()
    succeeded = False
    try:
        query_output = query_crew.kickoff(inputs=inputs)
        suggested_queries = parse_suggested_queries(query_output)
        search_results = run_product_searches(
            suggested_queries.queries,
            score_threshold=inputs["score_th"],
            max_total_results=max(inputs["top_recommendations_no"] * 2, 10),
        )
        products = run_product_scraping(
            search_results, max_products=inputs["top_recommendations_no"]
        )
        report_output = report_crew.kickoff(inputs={
            **inputs,
            "products_json": products.model_dump_json(indent=2),
            "company_context": about_company,
        })
        report_path = save_clean_html(report_output.raw)
        succeeded = True
        return {
            "query_count": len(suggested_queries.queries),
            "search_result_count": len(search_results.results),
            "product_count": len(products.products),
            "report_path": str(report_path.resolve()),
        }
    finally:
        if trace:
            try:
                agentops.end_trace(trace, end_state="Success" if succeeded else "Error")
            except Exception as error:
                print(f"AgentOps warning: {error}")

In [ ]:
# Start with five validated recommendations to keep API cost and execution time controlled.
workflow_summary = run_procurement_workflow({
    "product_name": "coffee machine for the office",
    "websites_list": ["www.amazon.eg", "www.jumia.com.eg", "www.noon.com/egypt-en"],
    "country_name": "Egypt",
    "no_keywords": 10,
    "language": "English",
    "score_th": 0.10,
    "top_recommendations_no": 5,
})

display(Markdown(
    f"""### Workflow completed\n
- Queries executed: **{workflow_summary['query_count']}**\n- Valid product pages found: **{workflow_summary['search_result_count']}**\n- Products extracted: **{workflow_summary['product_count']}**\n- Report: `{workflow_summary['report_path']}`"""
))